In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
import torch.nn.functional as F

# === Load datasets ===
imdb = load_dataset("imdb")
cmovie = {
    "train": load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train"),
    "test": load_dataset("cornell-movie-review-data/rotten_tomatoes", split="test")
}
# === Preprocessing ===
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === Tokenization & Vocab ===
def tokenize(text):
    return clean_text(text).split()

# Build vocab from IMDB
counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

# Only keep words with min_freq ≥ 5
vocab = {"<pad>": 0, "<unk>": 1}
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)
    ids = [vocab.get(token, UNK_IDX) for token in tokens[:max_len]]
    return torch.tensor(ids, dtype=torch.long)

# === Dataset Wrappers ===
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text_field = item["text"] if "text" in item else item["content"]
        tokens = text_pipeline(text_field, max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

In [8]:
train_loader_imdb = DataLoader(TextDataset(imdb["train"], lambda x: x), batch_size=64, shuffle=True, collate_fn=collate_batch)
train_loader_cmovie = DataLoader(TextDataset(cmovie["train"], lambda x: x), batch_size=256, shuffle=True, collate_fn=collate_batch)
test_loader_cmovie = DataLoader(TextDataset(cmovie["test"], lambda x: x), batch_size=256, shuffle=False, collate_fn=collate_batch)
# === Model ===
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextClassifier(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [9]:
# === Sliced Wasserstein ===
from sliced_wasserstein import sliced_wasserstein_distance

def feature_extractor(loader, max_samples=1000):
    features = []
    model.eval()
    sample_count = 0
    
    with torch.no_grad():
        for x, _ in loader:
            if sample_count >= max_samples:
                break
            x = x.to(device)
            feature = model(x)
            features.append(feature.cpu().numpy())
            sample_count += x.shape[0]
        return np.concatenate(features, axis=0)

imdb_features = feature_extractor(train_loader_imdb)
print("Extracted imdb features")

cmovie_features = feature_extractor(train_loader_cmovie)
print("Extracted cornell movie features")

print("Sliced Wasserstein Distance between cmovie and imdb:", 
      sliced_wasserstein_distance(cmovie_features, imdb_features))


# === Jensen-Shannon Divergence ===
from sklearn.feature_extraction.text import CountVectorizer

def KL_numpy(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    epsilon = 1e-10
    b = np.where(b == 0, epsilon, b)
    return np.sum(np.where(a != 0, a * np.log(a / b), 0))

imdb_small = load_dataset("imdb", split='train[:10000]')
cmovie_small = load_dataset("cornell-movie-review-data/rotten_tomatoes", split='train[:10000]')

texts_imdb = [x['text'] for x in imdb_small]
texts_cmovie = [x['text'] for x in cmovie_small]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_cmovie)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_cmovie = vectorizer.transform(texts_cmovie).toarray().astype(np.float32)

def to_prob_dist(data):
    data = data + 1e-10
    return data / data.sum(axis=1, keepdims=True)

datasets = {
    'IMDB': X_imdb,
    'CMovie': X_cmovie
}

prob_datasets = {}
avg_datasets = {}

for name, data in datasets.items():
    prob_datasets[name] = to_prob_dist(data)
    avg_datasets[name] = prob_datasets[name].mean(axis=0)

print("Jensen-Shannon Divergence between dataset pairs:")
for name1 in datasets.keys():
    for name2 in datasets.keys():
        if name1 != name2:
            m = (avg_datasets[name1] + avg_datasets[name2]) * 0.5
            js = 0.5 * KL_numpy(avg_datasets[name1], m) + 0.5 * KL_numpy(avg_datasets[name2], m)
            print(f"{name1} vs {name2}: {js:.12f}")


# === Gradient Cosine + L2 Norm ===
def get_gradient_vector(model, dataloader):
    model.train()
    model.zero_grad()
    x, y = next(iter(dataloader))
    x, y = x.to(device), y.to(device)

    out = model(x)
    loss = F.cross_entropy(out, y)
    loss.backward()

    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1))
    return torch.cat(grads)

grad_imdb = get_gradient_vector(model, train_loader_imdb)
grad_cmovie = get_gradient_vector(model, train_loader_cmovie)

l2_diff = torch.norm(grad_imdb - grad_cmovie).item()
cos_sim = F.cosine_similarity(grad_imdb.unsqueeze(0), grad_cmovie.unsqueeze(0)).item()

print(f"L2 norm of gradient difference: {l2_diff:.12f}")
print(f"Cosine similarity between gradients: {cos_sim:.12f}")


# === MMD ===
from discrepancies import MaximumMeanDiscrepancy
from kernels import LaplacianKernel

imdb_small = load_dataset("imdb", split='train[:1000]')
cmovie_small = load_dataset("cornell-movie-review-data/rotten_tomatoes", split='train[:1000]')

texts_imdb = [x['text'] for x in imdb_small]
texts_cmovie = [x['text'] for x in cmovie_small]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_cmovie)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_cmovie = vectorizer.transform(texts_cmovie).toarray().astype(np.float32)

mmd = MaximumMeanDiscrepancy(kernel=LaplacianKernel(1e-2))
print("MMD between imdb and cmovie:", mmd.compute(X_imdb, X_cmovie))


# === Hellinger Distance ===
def hellinger_np(p, q):
    return np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))

print("Hellinger Distance between imdb and cmovie:", hellinger_np(X_imdb, X_cmovie))


Extracted imdb features
Extracted cornell movie features
Sliced Wasserstein Distance between cmovie and imdb: 0.016718547056766973
Jensen-Shannon Divergence between dataset pairs:
IMDB vs CMovie: 0.104761724065
CMovie vs IMDB: 0.104761724065
L2 norm of gradient difference: 0.174406349659
Cosine similarity between gradients: 0.664377510548
MMD between imdb and cmovie: 0.2481556
Hellinger Distance between imdb and cmovie: 225.10898
